In [ ]:
import json

from pandas import concat, DataFrame, isna, read_csv
from scipy.stats import entropy

In [ ]:
report_name = "forward/combined_report_doe1215_threshold=0.8"
df_combined = read_csv(f"output/{report_name}.csv")

# json_index = df_combined[~isna(df_combined["probabilities"])].index
# df_combined.loc[json_index, "probabilities"] = df_combined.loc[json_index]["probabilities"].apply(json.loads)
df_combined["probabilities"] = df_combined["probabilities"].apply(json.loads)

# df_combined_metrics = read_csv(f"output/{report_name}_with_metrics.csv")
# df_combined_metrics["probabilities"] = df_combined_metrics["probabilities"].apply(
#     json.loads
# )

In [ ]:
def process(record):
    if isinstance(record, str):
        return json.loads(record)
    else:
        return record

df_combined["probabilities"] = df_combined["probabilities"].apply(process)

# Statistics/metrics

## Uniformity of probabilities

In [ ]:
records_new = []

# Skip scenarios for which already a record exists with metrics
for i in df_combined[
    ~df_combined["scenario_name"].isin(df_combined_metrics["scenario_name"].values)
].index:
    if not isinstance(df_combined.loc[i, "probabilities"], list):
        print(i)
        continue
    df_group = DataFrame(df_combined.loc[i, "probabilities"])

    n_targets = df_group["entity_target"].nunique()
    probabilities = df_group.groupby(["entity_source", "entity_target"])[
        "probability"
    ].sum()
    probabilities = probabilities.astype(float)

    kl_divergence = entropy(pk=probabilities.values, qk=[1 / n_targets] * n_targets)

    df_combined.loc[i, "kl_divergence"] = (
        kl_divergence  # closer to 0 is closer to uniform distribution
    )

    records_new.append(df_combined.loc[i].to_dict())

df_combined_metrics = concat([df_combined_metrics, DataFrame(records_new)]).reset_index(
    drop=True
)

### Number of merges

In [ ]:
df_combined_metrics["n_merges"] = df_combined_metrics["probabilities"].apply(
    lambda x: sum(r["n_merges"] for r in x)
)

### Uniformity of merges and splits

In [ ]:
def average_split_merge_kl(x):
    total = 0
    for r in x:
        if not r.get("split_merge_kl"):
            continue
        total += r["split_merge_kl"]

    return total / len(x)


df_combined_metrics["split_merge_kl"] = df_combined_metrics["probabilities"].apply(
    average_split_merge_kl
)

### Average number of resources (per production step)

In [ ]:
for scenario_name in df_combined_metrics["scenario_name"].unique():
    with open(f"{scenario_name}.json") as f:
        average_n_resources = (
            DataFrame(json.load(f)["production_resources"])
            .groupby("step")
            .size()
            .mean()
        )

    df_combined_metrics.loc[
        df_combined_metrics[
            df_combined_metrics["scenario_name"] == scenario_name
        ].index,
        "average_n_resources",
    ] = average_n_resources

### Average number of production steps executed before aggregations

In [ ]:
from numpy import mean

# Unpack lists and take calculate mean
df_combined_metrics["n_production_steps_aggregations"] = df_combined_metrics[
    "probabilities"
].apply(
    lambda x: mean(
        [i for r in x for l in r["n_production_steps_aggregations"] for i in l]
    )
)

In [ ]:
df_combined_metrics["probabilities"] = df_combined_metrics["probabilities"].apply(
    json.dumps
)
df_combined_metrics.to_csv(
    f"output/{report_name}_with_metrics.csv", index=False
)

# Visualize/process results

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

from pandas import concat, isna, notna, read_csv, set_option
from pandas.api.types import CategoricalDtype
from scipy.stats import pearsonr

set_option("display.max_colwidth", None)

In [ ]:
report_name = "forward/combined_report_doe1215_threshold=0.8"
df_combined_metrics = read_csv(f"output/{report_name}_with_metrics.csv")


def combine_steps(row):
    steps_order = ["WT", "DB", "WB", "Marking", "FT"]
    merge_steps = (
        row["merge_after_steps"].split("-") if notna(row["merge_after_steps"]) else []
    )
    split_steps = (
        row["split_after_steps"].split("-") if notna(row["split_after_steps"]) else []
    )

    combined = []
    for step in steps_order:
        if step in merge_steps:
            combined.append(f"{step}-merge")
        elif step in split_steps:
            combined.append(f"{step}-split")
        else:
            combined.append(f"{step}")

    return "-".join(combined)


def merge_split_order(row, first):
    after = "split" if first == "merge" else "merge"
    i_first = row.find(first)
    i_after = row.find(after)

    return (i_first != -1) & (i_first < i_after)


for parameter in [
    "merge_after_steps",
    "split_after_steps",
    "number_of_devices",
    "n_resources_factor",
]:
    df_combined_metrics[parameter] = df_combined_metrics["scenario_name"].str.extract(
        rf"{parameter}=([A-Za-z0-9-]+)"
    )


df_combined_metrics["merge_split_steps"] = df_combined_metrics.apply(combine_steps, axis=1)
df_combined_metrics["merge_before_split"] = df_combined_metrics["merge_split_steps"].apply(
    merge_split_order, first="merge"
)
df_combined_metrics["split_before_merge"] = df_combined_metrics["merge_split_steps"].apply(
    merge_split_order, first="split"
)
df_combined_metrics["count_merge"] = df_combined_metrics["merge_split_steps"].str.count(
    "merge"
)
df_combined_metrics["count_split"] = df_combined_metrics["merge_split_steps"].str.count(
    "split"
)


cat_types = {
    "number_of_devices": CategoricalDtype(
        categories=["50", "25-75", "10-90", "125-25", "10-170"], ordered=True
    ),
}
for c in df_combined_metrics.columns:
    if c in cat_types:
        df_combined_metrics[c] = df_combined_metrics[c].astype(cat_types[c])

In [ ]:
import pandas as pd
import statsmodels.api as sm
from sklearn.preprocessing import StandardScaler

y_variable = "n_recall_diff" #"kl_divergence"

# Prepare the data: drop rows with missing n_steps_diff or any factor
factors = [
    "split_before_merge",
    # "kl_divergence",
]
normalize_factors = [
    "count_merge",
    "count_split",
]
dummy_factors = [
    "number_of_devices",
    "n_resources_factor",
]
df_reg = df_combined_metrics.dropna(subset=factors + dummy_factors + [y_variable]).copy()

# Encode categorical variables
df_reg["split_before_merge"] = df_reg["split_before_merge"].astype(int)
df_reg["number_of_devices"] = df_reg["number_of_devices"].astype(str)
df_reg["n_resources_factor"] = df_reg["n_resources_factor"].astype(str)

# Normalize factors
scaler = StandardScaler()
for factor in normalize_factors:
    df_reg[factor] = scaler.fit_transform(df_reg[[factor]])

# Create dummy variables for categorical factors
dummy_factors = ["number_of_devices", "n_resources_factor"]
df_reg = pd.get_dummies(df_reg, columns=dummy_factors, drop_first=False, dtype=int)
df_reg.drop(
    ["number_of_devices_50", "n_resources_factor_1"],
    axis=1,
    inplace=True,
    errors="ignore",
)

dummy_columns = [c for c in df_reg.columns if any(f in c for f in dummy_factors)]

# Define X and y
X = df_reg[dummy_columns]
X = pd.concat([df_reg[factors+normalize_factors], X], axis=1)
y = df_reg[y_variable]

# Add constant for intercept
X = sm.add_constant(X)

# Fit the model
model = sm.OLS(y, X).fit()

# Print the summary
print(model.summary())

In [ ]:
# Get coefficients, confidence intervals, and p-values
coefs = model.params
conf = model.conf_int()
conf.columns = ["lower", "upper"]
pvalues = model.pvalues

variables = {
    "count_merge": "$|S_{merge}|$",
    "count_split": "$|S_{split}|$",
    "split_before_merge": "$f_{split-merge}$",
    "number_of_devices_25-75": "$D_{25-75}$",
    "number_of_devices_10-90": "$D_{10-90}$",
    "number_of_devices_125-25": "$D_{25-125}$",
    "number_of_devices_10-170": "$D_{10-170}$",
    "n_resources_factor_2": "$r_f=2$",
    "n_resources_factor_4": "$r_f=4$",
    "n_resources_factor_8": "$r_f=8$",
    "n_resources_factor_16": "$r_f=16$",
}

# Prepare DataFrame for plotting
coef_df = coefs.to_frame("coef").join(conf)
coef_df = coef_df.reset_index().rename(columns={"index": "variable"})
coef_df["pvalue"] = coef_df["variable"].map(pvalues)
coef_df.set_index("variable", inplace=True)

# Filter variables with p-value <= 0.005
significant_vars = [v for v in variables.keys() if coef_df.loc[v, "pvalue"] <= 0.005]

# Plot coefficients with confidence intervals for significant variables only
plt.figure(figsize=(8, 5))
plt.errorbar(
    significant_vars,
    coef_df.loc[significant_vars, "coef"],
    yerr=[
        coef_df.loc[significant_vars, "coef"] - coef_df.loc[significant_vars, "lower"],
        coef_df.loc[significant_vars, "upper"] - coef_df.loc[significant_vars, "coef"],
    ],
    fmt="o",
    capsize=5,
    color="b",
)
plt.axhline(0, color="grey", linestyle="--")

xtick_labels = [variables[l.get_text()] for l in plt.xticks()[1]]
plt.xticks(ticks=plt.xticks()[0], labels=xtick_labels, rotation=45, ha="right")

plt.ylabel("Coefficient")
plt.title("MLR Coefficients with 95% Confidence Intervals (p ≤ 0.005)")
plt.tight_layout()

plt.savefig(
    "output/forward/figures/forward-mlr-ols-coefficients-significant.svg", bbox_inches="tight"
)
plt.savefig(
    "output/forward/figures/forward-mlr-ols-coefficients-significant.eps", bbox_inches="tight"
)

In [ ]:
y_column = "n_recall_diff"

filter_scenario = "merge_after_steps=W[TB]\+"  # "split_after_steps=Marking"
df_plot = df_combined_metrics[
    df_combined_metrics["scenario_name"].str.contains(filter_scenario)
].copy()
# df_plot = df_combined_metrics.copy()

x_label = "split_after_steps"
y_labels = [y_column, "split_merge_kl"]
df_plot[x_label] = df_plot["scenario_name"].str.extract(rf"{x_label}=([A-Za-z0-9-]+)")

fig, axes = plt.subplots(nrows=len(y_labels), sharex=True, figsize=(20, 15))
sns.set_theme(font_scale=1)

# group_label = "number_of_devices"
group_label = "merge_after_steps"
df_plot[group_label] = df_plot["scenario_name"].str.extract(
    rf"{group_label}=([A-Za-z0-9-]+)"
)

for i, y_label in enumerate(y_labels):
    ax = sns.violinplot(
        data=df_plot[~isna(df_plot[y_column])],
        x=x_label,
        y=y_label,
        hue=group_label,
        ax=axes[i],
    )

    # ax.set_ylabel("$\delta_{fw}$")
    ax.tick_params(axis="x", rotation=90)
    ax.legend(loc="upper left", bbox_to_anchor=(1, 1))

# plt.grid()

# plt.savefig(
#     f"output/backward/figures/violin_{'+'.join(y_labels)}_x={x_label}_group={group_label}.svg",
#     bbox_inches="tight",
# )

In [ ]:
import ipywidgets as widgets
from IPython.display import display, clear_output

y_column = "n_recall_diff"

scenario_options = [
    "merge_after_steps",
    "number_of_devices",
    "n_resources_factor",
    "split_after_steps",
]

y_label_mapping = {
    y_column: "$n_{recall}$",
}

# Define interactive widgets
filter_scenario_widget = widgets.Text(
    value="",
    description="Filter Scenario:",
    layout=widgets.Layout(width="50%"),
)

x_label_widget = widgets.Dropdown(
    options=scenario_options,
    value="merge_after_steps",
    description="X-axis Label:",
)

y_labels_widget = widgets.SelectMultiple(
    options=df_combined_metrics.columns,
    value=[y_column, "split_merge_kl"],
    description="Y-axis Label:",
)

group_label_widget = widgets.Dropdown(
    options=scenario_options,
    value="merge_after_steps",
    description="Group Label:",
)


# Function to update the plot
def update_plot(filter_scenario, x_label, y_labels, group_label):
    clear_output(wait=True)
    display(
        filter_scenario_widget,
        x_label_widget,
        y_labels_widget,
        group_label_widget,
        plot_button,
    )

    df_plot = df_combined_metrics[
        df_combined_metrics["scenario_name"].str.contains(filter_scenario)
    ].copy()

    df_plot[x_label] = df_plot["scenario_name"].str.extract(
        rf"{x_label}=([A-Za-z0-9-]+)"
    )
    df_plot[group_label] = df_plot["scenario_name"].str.extract(
        rf"{group_label}=([A-Za-z0-9-]+)"
    )

    fig, axes = plt.subplots(nrows=len(y_labels), sharex=True, figsize=(20, 15))
    sns.set_theme(font_scale=1)

    for i, y_label in enumerate(y_labels):
        ax = sns.boxplot(
            data=df_plot[~isna(df_plot[y_column])],
            x=x_label,
            y=y_label,
            hue=group_label,
            ax=axes[i],
            legend="full",
        )
        ax.tick_params(axis="x", rotation=90)
        ax.set_ylabel(y_label_mapping.get(y_label, y_label))

    # plt.savefig(
    #     f"output/backward/figures/violin_{'+'.join(y_labels)}_x={x_label}_group={group_label}.svg",
    #     bbox_inches="tight",
    # )
    plt.show()


# Button to trigger the update
plot_button = widgets.Button(description="Update Plot")


# Event handler for the button
def on_button_click(b):
    update_plot(
        filter_scenario_widget.value,
        x_label_widget.value,
        y_labels_widget.value,
        group_label_widget.value,
    )


plot_button.on_click(on_button_click)

# Display widgets
display(
    filter_scenario_widget,
    x_label_widget,
    y_labels_widget,
    group_label_widget,
    plot_button,
)

In [ ]:
ax = df_combined_metrics.plot(kind="scatter", x="kl_divergence", y="n_recall_all", color="orange")
df_combined_metrics.plot(kind="scatter", x="kl_divergence", y="n_recall_threshold", marker=".", color="green", ax=ax)

In [ ]:
index = [
    "count_merge",
    "count_split",
    "merge_before_split",
]
columns = ["number_of_devices", "n_resources_factor"]

# Create a pivot table for interaction profile
interaction_profile = (
    df_combined_metrics.sort_values(index)
    .pivot_table(
        values="kl_divergence",
        index=index,
        columns=columns,
        aggfunc="mean",
    )
    .loc[:, (["50", "25-75", "10-90", "125-25", "10-170"], ["2", "4", "8", "16"])]
)

# Plot the heatmap
plt.figure(figsize=(20, 15))
sns.heatmap(
    interaction_profile,
    annot=True,
    fmt=".2f",
    cmap="coolwarm",
    cbar_kws={"label": "Mean kl_divergence"},
)
plt.title("Interaction Profile for KL divergence")
plt.xlabel("-".join(columns))
plt.ylabel("-".join(index))
plt.xticks(rotation=45, ha="right")
plt.tight_layout()

In [ ]:
import os
import sys

from pathlib import Path

basepath = Path(
    r"C:\Users\s158699\Documents\PhD\Experiments\Simulation\aggregated_event_data"
)

sys.path.append(os.path.join(*basepath.parts))

from aggregated_traces.utils.construct_ekg import (
    check_quantities,
    generate_networkx_di_graph,
    insert_DF_DP,
    insert_fractions,
    load_rdf_graph,
)

In [ ]:
from aggregated_traces.utils.visualization import load_rdf_graph

visualize_ekg_graph_rdf = load_rdf_graph(
    basepath.joinpath(
        "logs/scenarios/merge_after_steps=+number_of_devices=50+n_resources_factor=1+split_after_steps=WT-DB-WB-Marking-FT/run_0.json"
    ),
    store="Oxigraph",
)

# Insert DF/DP relations
visualize_ekg_graph_rdf = insert_DF_DP(visualize_ekg_graph_rdf)

# Insert fractions on the relations
# visualize_ekg_graph_rdf = insert_fractions(visualize_ekg_graph_rdf)

# Create NetworkX graph
# visualize_ekg_graph_nx = generate_networkx_di_graph(visualize_ekg_graph_rdf)

# generate_graph_visualization(visualize_ekg_graph_nx, base_figure_path="output/check")